# YTgist: Research Experiments & Benchmark Suite

This notebook provides interactive reproductions of the 5 core research experiments in **YTgist**:
- **Experiment 1**: Impact of Chunk Sizes & Overlap Ratios
- **Experiment 2**: Multi-Paradigm Summarization Comparison
- **Experiment 3**: Summary Length vs Quality & Pareto Frontier
- **Experiment 4**: Handling Long Transcripts (Scalability & Hierarchy)
- **Experiment 5**: Failure Mode Taxonomy & Error Analysis

---

## Setup & Environment Verification

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add workspace root to python path
workspace_root = os.path.abspath('..')
if workspace_root not in sys.path:
    sys.path.append(workspace_root)

import ytgist
print(f'YTgist Package Version: {ytgist.__version__}')

## Experiment 1: Chunk Sizes & Overlap Ratios
Evaluates chunk sizes (256, 512, 1024, 2048) and overlaps (0%, 10%, 20%).

In [ ]:
from experiments import run_exp1
import json

# Load or re-run Experiment 1
results_path = os.path.join('..', 'results', 'exp1_chunk_sizes.json')
if os.path.exists(results_path):
    with open(results_path) as f:
        exp1_data = json.load(f)['results']
else:
    exp1_data = run_exp1()['results']

df1 = pd.DataFrame(exp1_data)
display_cols = ['chunk_size_tokens', 'overlap_ratio', 'num_chunks', 'boundary_cutoffs_pct', 'rouge_1_f1', 'rouge_l_f1', 'latency_ms']
print(df1[display_cols].to_markdown(index=False))

# Plotting ROUGE-1 F1 across chunk sizes
plt.figure(figsize=(10, 5))
for ov in [0.0, 0.10, 0.20]:
    sub = df1[df1['overlap_ratio'] == ov]
    plt.plot(sub['chunk_size_tokens'], sub['rouge_1_f1'], marker='o', label=f'Overlap {int(ov*100)}%')

plt.title('Experiment 1: ROUGE-1 F1 vs. Chunk Size')
plt.xlabel('Chunk Size (tokens)')
plt.ylabel('ROUGE-1 F1')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## Experiment 2: Summarization Approaches Comparison
Compares Single-Shot, TF-IDF Salience, Map-Reduce, Iterative Refine, and Hierarchical Tree.

In [ ]:
results_path = os.path.join('..', 'results', 'exp2_summarization_approaches.json')
with open(results_path) as f:
    exp2_data = json.load(f)['results']

rows = []
for r in exp2_data:
    rows.append({
        'Approach': r['approach'],
        'Latency (ms)': r['latency_ms'],
        'ROUGE-1 F1': r['rouge_1']['f1'],
        'ROUGE-2 F1': r['rouge_2']['f1'],
        'ROUGE-L F1': r['rouge_l']['f1'],
        'Redundancy': r['redundancy_score']
    })
df2 = pd.DataFrame(rows)
print(df2.to_markdown(index=False))

# Bar chart of ROUGE-1 F1 and Latency
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

x = np.arange(len(df2))
width = 0.35

ax1.bar(x - width/2, df2['ROUGE-1 F1'], width, label='ROUGE-1 F1', color='#3b82f6')
ax2.bar(x + width/2, df2['Latency (ms)'], width, label='Latency (ms)', color='#f97316')

ax1.set_xticks(x)
ax1.set_xticklabels(df2['Approach'], rotation=25, ha='right')
ax1.set_ylabel('ROUGE-1 F1')
ax2.set_ylabel('Latency (ms)')
plt.title('Experiment 2: Summarization Approaches Performance')
plt.tight_layout()
plt.show()

## Experiment 3: Summary Length vs Quality & Pareto Frontier

In [ ]:
results_path = os.path.join('..', 'results', 'exp3_length_vs_quality.json')
with open(results_path) as f:
    exp3_data = json.load(f)['results']

df3 = pd.DataFrame(exp3_data)
print(df3[['sentence_budget', 'word_count', 'compression_ratio', 'rouge_1_precision', 'rouge_1_recall', 'rouge_1_f1', 'rouge_l_f1']].to_markdown(index=False))

# Plotting Precision vs Recall trade-off curve
plt.figure(figsize=(9, 5))
plt.plot(df3['word_count'], df3['rouge_1_precision'], marker='o', label='Precision', color='#ef4444')
plt.plot(df3['word_count'], df3['rouge_1_recall'], marker='s', label='Recall', color='#10b981')
plt.plot(df3['word_count'], df3['rouge_1_f1'], marker='^', label='F1 Score', color='#6366f1', linewidth=2)
plt.title('Experiment 3: Precision-Recall Trade-off across Summary Length')
plt.xlabel('Summary Word Budget')
plt.ylabel('Score')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## Experiment 4: Long Transcript Scalability

In [ ]:
results_path = os.path.join('..', 'results', 'exp4_long_transcripts.json')
with open(results_path) as f:
    exp4_data = json.load(f)['results']

df4 = pd.DataFrame(exp4_data)
print(df4[['corpus', 'word_count', 'strategy', 'latency_ms', 'rouge_1_f1', 'rouge_l_f1']].to_markdown(index=False))

# Latency scaling chart
plt.figure(figsize=(10, 5))
for strat in df4['strategy'].unique():
    sub = df4[df4['strategy'] == strat]
    plt.plot(sub['word_count'], sub['latency_ms'], marker='o', label=strat)

plt.title('Experiment 4: Processing Latency Scaling by Input Length')
plt.xlabel('Source Words')
plt.ylabel('Latency (ms)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

## Experiment 5: Failure Mode Taxonomy & Error Analysis

In [ ]:
results_path = os.path.join('..', 'results', 'exp5_error_analysis.json')
with open(results_path) as f:
    exp5_data = json.load(f)['taxonomy']

df5 = pd.DataFrame(exp5_data)
print(df5[['error_category', 'naive_pipeline_rate', 'ytgist_mitigated_rate', 'mitigation_strategy']].to_markdown(index=False))


## Interactive Video Gist Demo
Test YTgist end-to-end on any sample transcript or live YouTube video ID!

In [ ]:
from ytgist.data_loader import TranscriptLoader
from ytgist.preprocessing import clean_transcript, restore_punctuation_heuristic
from ytgist.summarizers import MapReduceSummarizer
from ytgist.extraction import KeyPhraseExtractor, TimelineExtractor

# Load sample transcript
items = TranscriptLoader.load_sample('short_5min')
cleaned_items = clean_transcript(items)
full_text = TranscriptLoader.to_plain_text(cleaned_items)
punctuated_text = restore_punctuation_heuristic(full_text)

# 1. Summarization
summarizer = MapReduceSummarizer(chunk_size=512, overlap_ratio=0.1)
summary = summarizer.summarize(punctuated_text, final_sentences=4)['summary']

# 2. Keyphrase extraction
phrases = KeyPhraseExtractor(max_phrases=8).extract(punctuated_text)

# 3. Key moments timeline
moments = TimelineExtractor(top_k=5).extract_from_transcript(cleaned_items)

print('=== FINAL STRUCTURED GIST ===\n')
print('SUMMARY:')
print(summary)
print('\nKEY TOPICS & PHRASES:')
for p, s in phrases:
    print(f'- {p} (score: {s:.2f})')
print('\nKEY MOMENTS TIMELINE:')
for m in moments:
    print(f'[{m.timestamp_str}] {m.text[:65]}...')